# Colab Z-Image Turbo + ngrok API

啟動 ComfyUI，並用 ngrok 暴露 Web UI 與 ComfyUI API。

In [ ]:
# 1. Install ComfyUI + dependencies
%cd /content
![ -d ComfyUI ] || git clone https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI
!pip -q install -r requirements.txt xformers pyngrok

In [ ]:
# 2. Download Z-Image Turbo models
%cd /content/ComfyUI
!mkdir -p models/diffusion_models models/text_encoders models/vae

!test -f models/diffusion_models/z_image_turbo_nvfp4.safetensors || wget -q --show-progress -O models/diffusion_models/z_image_turbo_nvfp4.safetensors https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/diffusion_models/z_image_turbo_nvfp4.safetensors
!test -f models/text_encoders/qwen_3_4b_fp4_mixed.safetensors || wget -q --show-progress -O models/text_encoders/qwen_3_4b_fp4_mixed.safetensors https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b_fp4_mixed.safetensors
!test -f models/vae/ae.safetensors || wget -q --show-progress -O models/vae/ae.safetensors https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors

In [ ]:
# 3. Start ngrok tunnel
from getpass import getpass
from pyngrok import ngrok

try:
    from google.colab import userdata
    token = userdata.get("NGROK_AUTHTOKEN")
except Exception:
    token = None

if not token:
    token = getpass("NGROK_AUTHTOKEN: ")

ngrok.set_auth_token(token)
ngrok.kill()

public_url = ngrok.connect(8188, "http").public_url
print("ComfyUI Web UI:", public_url)
print("ComfyUI API:", public_url + "/prompt")
print("History API:", public_url + "/history/{prompt_id}")

In [ ]:
# 4. Launch ComfyUI
%cd /content/ComfyUI
!python main.py --listen 0.0.0.0 --port 8188 --dont-upcast-attention --lowvram --enable-cors-header '*'

## 外部呼叫 API 範例
- 把 ComfyUI workflow 另存成 **API Format** 後，用 `/prompt` 提交；完成後用 `/history/{prompt_id}` 查結果。
- 將下載的 `image_z_image_turbo.json` 上傳到 ChatGPT/Claude/Gemini 等 AI 平台，讓 AI 幫你根據 json 內容來調整程式碼。

In [ ]:
import json, time, uuid, requests
from pathlib import Path

BASE = "https://你的-ngrok-url.ngrok-free.app"
HEADERS = {"ngrok-skip-browser-warning": "1"}

PROMPT = "a cute cat, cinematic light, high quality"
WIDTH = 1024
HEIGHT = 1024
BATCH_SIZE = 1
SEED = 123
STEPS = 8
CFG = 1

workflow = json.load(open("image_z_image_turbo.json", encoding="utf-8"))

workflow["57:27"]["inputs"]["text"] = PROMPT
workflow["57:13"]["inputs"]["width"] = WIDTH
workflow["57:13"]["inputs"]["height"] = HEIGHT
workflow["57:13"]["inputs"]["batch_size"] = BATCH_SIZE
workflow["57:3"]["inputs"]["seed"] = SEED
workflow["57:3"]["inputs"]["steps"] = STEPS
workflow["57:3"]["inputs"]["cfg"] = CFG

prompt_id = requests.post(
    f"{BASE}/prompt",
    json={"prompt": workflow, "client_id": str(uuid.uuid4())},
    headers=HEADERS,
).json()["prompt_id"]

while True:
    history = requests.get(f"{BASE}/history/{prompt_id}", headers=HEADERS).json()
    if prompt_id in history:
        break
    time.sleep(2)

Path("outputs").mkdir(exist_ok=True)

for output in history[prompt_id]["outputs"].values():
    for img in output.get("images", []):
        data = requests.get(f"{BASE}/view", params=img, headers=HEADERS).content
        path = Path("outputs") / img["filename"]
        path.write_bytes(data)
        print("saved:", path)